In [1]:
# Q2: MLP vs CNN on CIFAR-10
# Q2(a): Clean images
# Q2(b): Spatially permuted images (fixed permutation, RGB pixels kept together)
# Q2(c): Central 8x8 occlusion
# Install: pip install torch torchvision matplotlib numpy scikit-learn
# Run: python q2_cnn_mlp.py

import random, json, csv
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

SEED=26001
BATCH=128
EPOCHS=20
LR=0.001
OUT=Path("q2_results")
OUT.mkdir(exist_ok=True)

random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])
base_train=datasets.CIFAR10("./data",train=True,download=True,transform=transform)
base_test=datasets.CIFAR10("./data",train=False,download=True,transform=transform)
labels=np.asarray(base_train.targets)
train_idx,val_idx=train_test_split(np.arange(len(labels)),test_size=5000,
                                  stratify=labels,random_state=SEED)
train_ds=Subset(base_train,train_idx)
val_ds=Subset(base_train,val_idx)
test_ds=base_test

class TransformDataset(Dataset):
    def __init__(self,subset,mode):
        self.subset=subset
        self.mode=mode
        g=torch.Generator().manual_seed(SEED)
        self.permutation=torch.randperm(32*32,generator=g)
    def __len__(self):return len(self.subset)
    def __getitem__(self,i):
        x,y=self.subset[i]
        if self.mode=="shuffle":
            # Shuffle spatial locations consistently; RGB channels stay together.
            x=x.reshape(3,1024)[:,self.permutation].reshape(3,32,32)
        elif self.mode=="occlusion":
            # On normalized data, zero represents mean-colour masking.
            x=x.clone()
            x[:,12:20,12:20]=0
        return x,y

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(3072,512),nn.ReLU(),
                               nn.Linear(512,256),nn.ReLU(),
                               nn.Linear(256,10))
    def forward(self,x):return self.net(x.flatten(1))

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.ReLU(),
            nn.Conv2d(32,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),
            nn.Conv2d(64,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.MaxPool2d(2)
        )
        self.classifier=nn.Sequential(nn.Flatten(),nn.Linear(128*4*4,256),
                                      nn.ReLU(),nn.Dropout(0.30),nn.Linear(256,10))
    def forward(self,x):return self.classifier(self.features(x))

def nparams(m):return sum(p.numel() for p in m.parameters() if p.requires_grad)

def loaders(tr,va,te):
    return (DataLoader(tr,batch_size=BATCH,shuffle=True,num_workers=2),
            DataLoader(va,batch_size=BATCH,shuffle=False,num_workers=2),
            DataLoader(te,batch_size=BATCH,shuffle=False,num_workers=2))

@torch.no_grad()
def evaluate(model,loader):
    model.eval();fn=nn.CrossEntropyLoss()
    total=correct=loss_sum=0
    for x,y in loader:
        x,y=x.to(device),y.to(device)
        z=model(x);loss=fn(z,y)
        loss_sum+=loss.item()*y.size(0)
        correct+=(z.argmax(1)==y).sum().item()
        total+=y.size(0)
    return loss_sum/total,correct/total

def plot(h,name):
    plt.figure()
    plt.plot(h["train_acc"],label="Train accuracy")
    plt.plot(h["val_acc"],label="Validation accuracy")
    plt.xlabel("Epoch");plt.ylabel("Accuracy");plt.legend();plt.grid(True)
    plt.savefig(OUT/f"{name}_accuracy.png",dpi=200);plt.close()
    plt.figure()
    plt.plot(h["train_loss"],label="Train loss")
    plt.plot(h["val_loss"],label="Validation loss")
    plt.xlabel("Epoch");plt.ylabel("Loss");plt.legend();plt.grid(True)
    plt.savefig(OUT/f"{name}_loss.png",dpi=200);plt.close()

def run(model,name,tr,va,te):
    model.to(device)
    opt=torch.optim.Adam(model.parameters(),lr=LR)
    fn=nn.CrossEntropyLoss()
    h={"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[]}
    print("\\n",name,"parameters:",nparams(model))
    for epoch in range(EPOCHS):
        model.train();total=correct=loss_sum=0
        for x,y in tr:
            x,y=x.to(device),y.to(device)
            opt.zero_grad();z=model(x);loss=fn(z,y)
            loss.backward();opt.step()
            loss_sum+=loss.item()*y.size(0)
            correct+=(z.argmax(1)==y).sum().item();total+=y.size(0)
        tr_loss,tr_acc=loss_sum/total,correct/total
        vl,va_acc=evaluate(model,va)
        h["train_loss"].append(tr_loss);h["train_acc"].append(tr_acc)
        h["val_loss"].append(vl);h["val_acc"].append(va_acc)
        print(f"{name} Epoch {epoch+1}/{EPOCHS}: train acc={tr_acc:.4f}, val acc={va_acc:.4f}")
    test_loss,test_acc=evaluate(model,te)
    h.update({"test_loss":test_loss,"test_acc":test_acc,"parameters":nparams(model)})
    with open(OUT/f"{name}_history.json","w") as f:json.dump(h,f)
    torch.save(model.state_dict(),OUT/f"{name}.pt")
    plot(h,name)
    return {"experiment":name,"parameters":nparams(model),
            "best_val_acc":max(h["val_acc"]),"test_acc":test_acc,
            "test_loss":test_loss}

results=[]

# Q2(a): Clean condition
tr,va,te=loaders(train_ds,val_ds,test_ds)
results.append(run(MLP(),"Q2a_MLP_clean",tr,va,te))
tr,va,te=loaders(train_ds,val_ds,test_ds)
results.append(run(CNN(),"Q2a_CNN_clean",tr,va,te))

# Q2(b): Spatial permutation applied to training, validation, and test data
s_train=TransformDataset(train_ds,"shuffle")
s_val=TransformDataset(val_ds,"shuffle")
s_test=TransformDataset(test_ds,"shuffle")
tr,va,te=loaders(s_train,s_val,s_test)
results.append(run(MLP(),"Q2b_MLP_spatial_shuffle",tr,va,te))
tr,va,te=loaders(s_train,s_val,s_test)
results.append(run(CNN(),"Q2b_CNN_spatial_shuffle",tr,va,te))

# Q2(c): Central occlusion applied to training, validation, and test data
o_train=TransformDataset(train_ds,"occlusion")
o_val=TransformDataset(val_ds,"occlusion")
o_test=TransformDataset(test_ds,"occlusion")
tr,va,te=loaders(o_train,o_val,o_test)
results.append(run(MLP(),"Q2c_MLP_occlusion",tr,va,te))
tr,va,te=loaders(o_train,o_val,o_test)
results.append(run(CNN(),"Q2c_CNN_occlusion",tr,va,te))

with open(OUT/"results.csv","w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=results[0].keys())
    w.writeheader();w.writerows(results)
print("\\nQ2 complete. Results and curves saved to:",OUT.resolve())


Device: cpu
\n Q2a_MLP_clean parameters: 1707274
Q2a_MLP_clean Epoch 1/20: train acc=0.4155, val acc=0.4556
Q2a_MLP_clean Epoch 2/20: train acc=0.4904, val acc=0.4790
Q2a_MLP_clean Epoch 3/20: train acc=0.5285, val acc=0.5042
Q2a_MLP_clean Epoch 4/20: train acc=0.5612, val acc=0.5094
Q2a_MLP_clean Epoch 5/20: train acc=0.5896, val acc=0.5194
Q2a_MLP_clean Epoch 6/20: train acc=0.6168, val acc=0.5258
Q2a_MLP_clean Epoch 7/20: train acc=0.6406, val acc=0.5252
Q2a_MLP_clean Epoch 8/20: train acc=0.6656, val acc=0.5214
Q2a_MLP_clean Epoch 9/20: train acc=0.6906, val acc=0.5280
Q2a_MLP_clean Epoch 10/20: train acc=0.7111, val acc=0.5214
Q2a_MLP_clean Epoch 11/20: train acc=0.7339, val acc=0.5232
Q2a_MLP_clean Epoch 12/20: train acc=0.7572, val acc=0.5168
Q2a_MLP_clean Epoch 13/20: train acc=0.7726, val acc=0.5128
Q2a_MLP_clean Epoch 14/20: train acc=0.7942, val acc=0.5120
Q2a_MLP_clean Epoch 15/20: train acc=0.8052, val acc=0.5142
Q2a_MLP_clean Epoch 16/20: train acc=0.8208, val acc=0.5146
